# UK Energy Price Forecasting — Colab training (real data)

**Steps:** (1) Runtime → Change runtime type → **T4 GPU**. (2) Run all cells. (3) When prompted, upload **real_panel.parquet**. (4) Download **ablation_real.csv** at the end and send it back.

In [ ]:
!pip -q install "darts>=0.30" "chronos-forecasting" 2>/dev/null
!git clone -q https://github.com/elyokerr/Projects.git
import sys; sys.path.insert(0, "/content/Projects/uk-energy-price-forecasting")
print("setup done")

### Upload the data
Upload **real_panel.parquet** from your laptop, found at `uk-energy-price-forecasting/data/processed/real_panel.parquet`.

In [ ]:
from google.colab import files
up = files.upload()  # pick real_panel.parquet
import io, pandas as pd
fname = list(up)[0]
open("real_panel.parquet","wb").write(up[fname])
from src.build.fixtures import load_panel_from_parquet
bundle = load_panel_from_parquet("real_panel.parquet")
print("panel loaded, target len:", len(bundle.target))

### Build rolling origins (last ~30 days)

In [ ]:
from src.backtest.rolling_origin import generate_origins, run_backtest, build_ablation_table
idx = bundle.target.time_index
origins = generate_origins(idx, start=idx[-31*48])
print(len(origins), "origins")

### Train + backtest every model on identical origins
LightGBM + TiDE + TFT are trained once (refit=False). Chronos is zero-shot.

In [ ]:
from src.models.baselines import SeasonalNaive
from src.models.global_ml import GlobalLGBM
from src.models.global_dl import GlobalTiDE, GlobalTFT
results = {}
results["seasonal_naive"] = run_backtest(SeasonalNaive(), bundle, origins, horizon=48, model_name="seasonal_naive")
print("naive done")
results["global_lgbm"] = run_backtest(GlobalLGBM(), bundle, origins, horizon=48, model_name="global_lgbm")
print("lgbm done")
import gc, torch
results["global_tide"] = run_backtest(GlobalTiDE(n_epochs=30), bundle, origins, horizon=48, model_name="global_tide")
gc.collect(); torch.cuda.empty_cache(); print("tide done")
results["global_tft"] = run_backtest(GlobalTFT(n_epochs=30), bundle, origins, horizon=48, model_name="global_tft")
gc.collect(); torch.cuda.empty_cache(); print("tft done")

### Chronos zero-shot (optional — skip if it errors)

In [ ]:
try:
    import numpy as np
    from src.models.foundation import chronos_forecast
    # build a simple per-origin chronos backtest matching the harness output
    from src.backtest.rolling_origin import BacktestResult
    acts, fc = [], {0.1:[],0.5:[],0.9:[]}
    vals = bundle.target.values().flatten(); ti = bundle.target.time_index
    for o in origins:
        pos = ti.get_loc(o)
        ctx = vals[:pos+1]
        q = chronos_forecast(ctx, horizon=48, quantiles=(0.1,0.5,0.9))
        for k in fc: fc[k].append(q[k])
        acts.append(vals[pos+1:pos+49])
    import numpy as np
    results["chronos_zeroshot"] = BacktestResult("chronos_zeroshot", origins, (0.1,0.5,0.9),
        np.stack(acts), {k:np.stack(v) for k,v in fc.items()}, 48)
    print("chronos done")
except Exception as e:
    print("chronos skipped:", e)

### Ablation table + download

In [ ]:
table = build_ablation_table(results, baseline="seasonal_naive").round(3)
print(table.to_string())
table.to_csv("ablation_real.csv")
from google.colab import files
files.download("ablation_real.csv")